In [1]:
# @title Prepare Environment

# Установки для VBench
!pip install -q decord openai-clip timm fvcore iopath yacs fairscale pyiqa imageio-ffmpeg
!test -d /content/VBench || git clone https://github.com/Vchitect/VBench.git
import sys; sys.path.insert(0, "/content/VBench")

# Установки для FVD
!pip install scipy pandas opencv-python imageio-ffmpeg
!git clone https://github.com/piergiaj/pytorch-i3d.git || true

# Установки для основного пайплайна
!pip install torch==2.6
!pip install torchvision==0.21
!pip install torchsde einops diffusers accelerate
!pip install -q av gguf triton sageattention

!git clone https://github.com/Isi-dev/ComfyUI /content/ComfyUI || true
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git /content/ComfyUI/custom_nodes/ComfyUI_GGUF || true
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI_GGUF/requirements.txt
!apt -y install -qq aria2 ffmpeg

!mkdir -p /content/ComfyUI/models/unet
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae
!mkdir -p /content/ComfyUI/models/clip_vision
!mkdir -p /content/ComfyUI/input
!mkdir -p /content/ComfyUI/output

# Клонируем репозиторий проекта (нужен для src/)
!git clone https://github.com/REDISKA3000/ftiad_cursach.git /content/ftiad_cursach || git -C /content/ftiad_cursach pull

# ---------- Скачиваем модели ----------
useQ6 = False  # @param {"type":"boolean"}

if useQ6:
    unet_filename = "wan2.2-i2v-rapid-aio-v10-nsfw-Q6_K.gguf"
else:
    unet_filename = "wan2.2-i2v-rapid-aio-v10-nsfw-Q4_K.gguf"

!hf download befox/WAN2.2-14B-Rapid-AllInOne-GGUF \
    v10/{unet_filename} \
    --local-dir /content/ComfyUI/models/unet
!mv /content/ComfyUI/models/unet/v10/{unet_filename} \
    /content/ComfyUI/models/unet/{unet_filename} 2>/dev/null || true

unet_name = unet_filename

import os
unet_path = f"/content/ComfyUI/models/unet/{unet_name}"
assert os.path.exists(unet_path), f"UNet not downloaded: {unet_path}"
print(f"✅ UNet OK: {os.path.getsize(unet_path) / 1e9:.1f} GB")

!hf download NSFW-API/NSFW-Wan-UMT5-XXL \
    nsfw_wan_umt5-xxl_fp8_scaled.safetensors \
    --local-dir /content/ComfyUI/models/text_encoders
!mv /content/ComfyUI/models/text_encoders/nsfw_wan_umt5-xxl_fp8_scaled.safetensors \
    /content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors 2>/dev/null || true

!hf download Comfy-Org/Wan_2.1_ComfyUI_repackaged \
    split_files/vae/wan_2.1_vae.safetensors \
    --local-dir /content/ComfyUI/models/vae
!mv /content/ComfyUI/models/vae/split_files/vae/wan_2.1_vae.safetensors \
    /content/ComfyUI/models/vae/wan_2.1_vae.safetensors 2>/dev/null || true

!hf download Comfy-Org/Wan_2.1_ComfyUI_repackaged \
    split_files/clip_vision/clip_vision_h.safetensors \
    --local-dir /content/ComfyUI/models/clip_vision
!mv /content/ComfyUI/models/clip_vision/split_files/clip_vision/clip_vision_h.safetensors \
    /content/ComfyUI/models/clip_vision/clip_vision_h.safetensors 2>/dev/null || true

print("✅ Environment prepared")
print("UNet:", unet_name)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 17.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 20.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
import os
import sys
import json
import shutil
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import cv2
import imageio
import matplotlib.pyplot as plt

from PIL import Image
from scipy import linalg
from IPython.display import display, HTML, Image as IPImage

# Если запускается в Colab
try:
    from google.colab import files
except ImportError:
    files = None

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True")

# Пути должны быть добавлены ДО импортов ComfyUI
COMFY_ROOT = Path("/content/ComfyUI")
# Добавляем репозиторий проекта в путь (для from src.metrics... imports)
WORLD_MODEL_ROOT = Path("/content/ftiad_cursach")
if not WORLD_MODEL_ROOT.exists():
    raise FileNotFoundError(f"ftiad_cursach repo not found: {WORLD_MODEL_ROOT}. Run cell 0 first.")
if str(WORLD_MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(WORLD_MODEL_ROOT))


if not COMFY_ROOT.exists():
    raise FileNotFoundError(f"ComfyUI not found: {COMFY_ROOT}")


if str(COMFY_ROOT) not in sys.path:
    sys.path.insert(0, str(COMFY_ROOT))


# Импорты ComfyUI
from comfy_extras.nodes_wan import WanImageToVideo
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from nodes import (
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    LoadImage,
    CLIPVisionLoader,
    CLIPVisionEncode,
)
from comfy import model_management



PYTORCH_CUDA_ALLOC_CONF set to expandable_segments:True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# World Metadata


## World Spec

In [5]:
# ===== Extended world_spec block =====

from copy import deepcopy
from pathlib import Path


DEFAULT_CAMERA_PROMPTS = [
    "static camera",
    "slow zoom in",
    "pan left",
    "slight forward motion",
]


def build_world_spec(
    image_path: str,
    next_scene_prompt: str,
    camera_prompt: str = "static camera",
    target_objects=None,
    main_subject: str = None,
    secondary_subjects=None,
    expected_action: str = None,
    scene_context: str = None,
    style_tags=None,
    consistency_constraints=None,
    chunk_idx: int = 0,
    previous_chunk_last_frame: str = None,
    model_name: str = "custom_i2v_model",
    model_type: str = "videogen",
    generate_type: str = "i2v",
    frames_per_chunk: int = 40,
    fps: int = 8,
    resolution=(320, 512),
    num_chunks: int = 3,
    camera_trajectory_gt=None,
    camera_transform_hint=None,
    motion_mask_hint=None,
    expected_scene_transition: str = None,
    chunk_prompt_schedule=None,
    entity_region_hints=None,
    affected_entities=None,
    unaffected_entities=None,
    expected_motion=None,
    event_sequence=None,
    revisit_pairs=None,
):
    if target_objects is None:
        target_objects = []
    if secondary_subjects is None:
        secondary_subjects = []
    if style_tags is None:
        style_tags = []
    if consistency_constraints is None:
        consistency_constraints = [
            "keep the same subject identity",
            "keep the same scene",
            "avoid sudden scene changes",
            "avoid extra objects",
        ]
    if chunk_prompt_schedule is None:
        chunk_prompt_schedule = {}
    if entity_region_hints is None:
        entity_region_hints = {}

    resolution = list(resolution)

    layout_spec = {
        "camera_prompt": camera_prompt,
        "camera_instruction_text": camera_prompt,
        "camera_trajectory_gt": camera_trajectory_gt,
        "camera_transform_hint": camera_transform_hint,
    }

    world_spec = {
        # ---------- rollout bookkeeping ----------
        "chunk_idx": chunk_idx,
        "num_chunks": num_chunks,
        "current_scene_image": str(image_path),
        "previous_chunk_last_frame": previous_chunk_last_frame,

        # ---------- model metadata ----------
        "model_name": model_name,
        "model_type": model_type,
        "generate_type": generate_type,

        # ---------- generation config ----------
        "frames_per_chunk": frames_per_chunk,
        "fps": fps,
        "resolution": resolution,

        # ---------- semantic specification ----------
        "next_scene_prompt": next_scene_prompt,
        "expected_scene_transition": expected_scene_transition,
        "camera_prompt": camera_prompt,
        "camera_instruction_text": camera_prompt,
        "layout_spec": layout_spec,
        "camera_trajectory_gt": camera_trajectory_gt,
        "camera_transform_hint": camera_transform_hint,
        "motion_mask_hint": motion_mask_hint,
        "entity_region_hints": deepcopy(entity_region_hints),

        "main_subject": main_subject,
        "secondary_subjects": secondary_subjects,
        "target_objects": target_objects,
        "expected_action": expected_action,
        "scene_context": scene_context,
        "style_tags": style_tags,
        "consistency_constraints": consistency_constraints,
        "chunk_prompt_schedule": chunk_prompt_schedule,
        "affected_entities": deepcopy(affected_entities),
        "unaffected_entities": deepcopy(unaffected_entities),
        "expected_motion": deepcopy(expected_motion),
        "event_sequence": deepcopy(event_sequence),
        "revisit_pairs": deepcopy(revisit_pairs),
    }

    return world_spec


## Сборка positive_prompt из world_spec

In [6]:
# ===== Build positive prompt from extended world_spec =====

def _list_to_text(values):
    if not values:
        return ""
    if len(values) == 1:
        return values[0]
    return ", ".join(values[:-1]) + f" and {values[-1]}"


def _get_chunk_prompt(world_spec: dict):
    chunk_idx = world_spec.get("chunk_idx", 0)
    schedule = world_spec.get("chunk_prompt_schedule", {}) or {}
    if chunk_idx in schedule:
        return schedule[chunk_idx]
    return world_spec.get("next_scene_prompt", "")


def build_world_prompt(world_spec: dict):
    parts = []

    base_prompt = _get_chunk_prompt(world_spec)
    if base_prompt:
        parts.append(base_prompt.strip().rstrip("."))

    if world_spec.get("scene_context"):
        parts.append(f"scene context: {world_spec['scene_context']}")

    if world_spec.get("main_subject"):
        parts.append(f"main subject: {world_spec['main_subject']}")

    secondary = world_spec.get("secondary_subjects", [])
    if secondary:
        parts.append(f"secondary subjects: {_list_to_text(secondary)}")

    target_objects = world_spec.get("target_objects", [])
    if target_objects:
        parts.append(
            f"target objects visible in scene: {_list_to_text(target_objects)}")

    if world_spec.get("expected_action"):
        parts.append(f"expected action: {world_spec['expected_action']}")

    if world_spec.get("expected_scene_transition"):
        parts.append(
            f"expected scene transition: {world_spec['expected_scene_transition']}")

    style_tags = world_spec.get("style_tags", [])
    if style_tags:
        parts.append(f"style: {_list_to_text(style_tags)}")

    if world_spec.get("camera_instruction_text"):
        parts.append(f"camera: {world_spec['camera_instruction_text']}")

    constraints = world_spec.get("consistency_constraints", [])
    if constraints:
        parts.append("constraints: " + "; ".join(constraints))

    final_prompt = ". ".join(parts).strip()
    if not final_prompt.endswith("."):
        final_prompt += "."
    return final_prompt

## Обновление world_spec после каждого чанка

In [8]:
# ===== Update extended world_spec after each chunk =====

def update_world_spec_after_chunk(
    world_spec: dict,
    last_frame_path: str,
    chunk_idx: int,
):
    new_spec = deepcopy(world_spec)
    new_spec["chunk_idx"] = chunk_idx
    new_spec["previous_chunk_last_frame"] = world_spec.get("current_scene_image")
    new_spec["current_scene_image"] = str(last_frame_path)
    return new_spec


# ===== Debug preview =====

def preview_world_spec(world_spec: dict):
    preview_keys = [
        "chunk_idx",
        "num_chunks",
        "model_name",
        "model_type",
        "generate_type",
        "current_scene_image",
        "previous_chunk_last_frame",
        "next_scene_prompt",
        "camera_prompt",
        "camera_instruction_text",
        "main_subject",
        "secondary_subjects",
        "target_objects",
        "expected_action",
        "scene_context",
        "style_tags",
        "frames_per_chunk",
        "fps",
        "resolution",
        "layout_spec",
        "motion_mask_hint",
        "camera_trajectory_gt",
        "camera_transform_hint",
        "event_sequence",
        "revisit_pairs",
    ]
    return {k: world_spec.get(k) for k in preview_keys}


# Omni-Metric

Core Omni metric implementations now live in `src/metrics/omni/`.
The notebook keeps only lightweight orchestration, logging helpers, and the optional CLIP-vision embedding backend passed into the modular runner.


In [9]:
# ===== Omni notebook-side imports =====

from src.metrics.omni.common.results import (
    OMNI_CLIP_LOG_COLUMNS,
    OMNI_IMPLEMENTATION_STATUS,
    OMNI_SCORE_KEYS,
    OMNI_VISUAL_STACK_KEYS,
    omni_jsonable as _omni_jsonable,
)
from src.metrics.omni.summary import add_optional_full_rollout_omni_summary

# Core Omni helpers now live in src/metrics/omni/.
# Notebook keeps only orchestration, config, logging, and visualization.


# VBench

В проекте VBench используется для оценки качества каждого сгенерированного чанка и полного итогового rollout-видео.

Мы считаем 6 метрик:

- **Subject Consistency** — насколько стабильно главный объект сохраняет внешний вид внутри чанка
- **Background Consistency** — насколько стабильно сохраняется фон и сцена
- **Motion Smoothness** — насколько плавным и естественным выглядит движение
- **Dynamic Degree** — насколько в видео вообще присутствует заметная динамика
- **Aesthetic Quality** — общая визуальная привлекательность кадров
- **Imaging Quality** — техническое качество изображения: резкость, шум, артефакты, пересветы

Метрики считаются:
- после генерации каждого чанка
- сохраняются вместе с `chunk_idx` в историю и CSV
- затем используются для построения графиков по номеру чанка
- дополнительно считаются для полного rollout-видео

Это позволяет отслеживать, ухудшается ли качество генерации от чанка к чанку и как ведет себя итоговое длинное видео.


In [ ]:
from src.metrics.vbench import DEFAULT_VBENCH_DIMENSIONS, VBenchMetric, summarize_chunk_vbench_metrics

VBENCH_CUSTOM_DIMS = list(DEFAULT_VBENCH_DIMENSIONS)


def save_chunk_metrics_csv(clip_log, out_csv_path: str):
    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_new = pd.DataFrame(clip_log)

    if out_csv_path.exists() and out_csv_path.stat().st_size > 0:
        df_existing = pd.read_csv(out_csv_path)
        key_cols = [c for c in ("sample_id", "chunk_idx") if c in df_existing.columns and c in df_new.columns]
        if key_cols:
            existing_keys = set(zip(*[df_existing[c].tolist() for c in key_cols]))
            new_keys = set(zip(*[df_new[c].tolist() for c in key_cols]))
            mask = pd.Series(
                [tuple(row[c] for c in key_cols) in new_keys for _, row in df_existing.iterrows()]
            )
            df_existing = df_existing[~mask.values]
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_out = df_new

    df_out.to_csv(out_csv_path, index=False)
    return str(out_csv_path), df_out


def save_rollout_metrics_csv(rollout_record: dict, out_csv_path: str):
    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_new = pd.DataFrame([rollout_record])

    if out_csv_path.exists() and out_csv_path.stat().st_size > 0:
        df_existing = pd.read_csv(out_csv_path)
        if "sample_id" in df_existing.columns:
            df_existing = df_existing[df_existing["sample_id"] != rollout_record.get("sample_id")]
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_out = df_new

    df_out.to_csv(out_csv_path, index=False)
    return str(out_csv_path), df_out


def save_attempt_log_csv(attempts_log, out_csv_path: str):
    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    if not attempts_log:
        return str(out_csv_path), pd.DataFrame()
    df_new = pd.DataFrame(attempts_log)
    if out_csv_path.exists() and out_csv_path.stat().st_size > 0:
        df_existing = pd.read_csv(out_csv_path)
        key_cols = [c for c in ("sample_id", "chunk_idx", "attempt")
                    if c in df_existing.columns and c in df_new.columns]
        if key_cols:
            new_keys = set(zip(*[df_new[c].tolist() for c in key_cols]))
            mask = pd.Series(
                [tuple(row[c] for c in key_cols) in new_keys for _, row in df_existing.iterrows()]
            )
            df_existing = df_existing[~mask.values]
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_out = df_new
    df_out.to_csv(out_csv_path, index=False)
    return str(out_csv_path), df_out


# FVD

## FVD через `src/metrics/fvd`


In [13]:
from src.metrics.fvd import FVDMetric

fvd_metric = FVDMetric(
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=4,
    num_frames=16,
    resize_to=(224, 224),
    sampling_strategy="uniform",
    max_pairs=None,
    cache_features=False,
    cache_dir=None,
    verbose=True,
)


# Основной пайплайн

In [16]:


from src.metrics.omni import CameraControlMetric, InterStabLMetric, InterStabNMetric, ObjectControlMetric, OmniMetricRunner, TransitionsDetectMetric, build_omni_spec as modular_build_omni_spec, configure_omni_clip_backend, frame_embedding

# ---------- init lightweight node wrappers ----------
unet_loader = UnetLoaderGGUF()
model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
clip_vision_loader = CLIPVisionLoader()
clip_vision_encode = CLIPVisionEncode()
configure_omni_clip_backend(
    loader=clip_vision_loader,
    encoder=clip_vision_encode,
    reset_model_cache=True,
)
load_image = LoadImage()
wan_image_to_video = WanImageToVideo()
ksampler = KSampler()
vae_decode = VAEDecode()


# ---------- memory ----------
def clear_memory(aggressive: bool = True):
    gc.collect()

    try:
        if aggressive:
            model_management.unload_all_models()
        if hasattr(model_management, "cleanup_models_gc"):
            model_management.cleanup_models_gc()
        else:
            model_management.cleanup_models()
    except Exception:
        pass

    try:
        model_management.soft_empty_cache()
    except Exception:
        pass

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    gc.collect()


omni_runner = OmniMetricRunner(
    interstab_l_metric=InterStabLMetric(
        embedding_backend=frame_embedding,
        static_threshold=0.08,
        fallback_pair_strategy="long_range",
        verbose=False,
    ),
    interstab_n_metric=InterStabNMetric(
        flow_backend=None,
        beta=1.25,
        region_padding=0.0,
        verbose=False,
    ),
    intercov_metric=None,
    interorder_metric=None,
    transitions_detect_metric=TransitionsDetectMetric(
        threshold=None,
        min_scene_length=1,
        smoothing_window=None,
        verbose=False,
    ),
    object_control_metric=ObjectControlMetric(
        object_verifier=None,
        embedding_backend=frame_embedding,
        verbose=False,
    ),
    camera_control_metric=CameraControlMetric(
        camera_verifier=None,
        flow_backend=None,
        verbose=False,
    ),
    verbose=False,
)


# ---------- io ----------
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def upload_image():
    ensure_dir("/content/ComfyUI/input")
    uploaded = files.upload()

    for filename in uploaded.keys():
        src_path = f"/content/{filename}"
        dest_path = f"/content/ComfyUI/input/{filename}"
        shutil.move(src_path, dest_path)
        print(f"✅ Image saved to: {dest_path}")
        return dest_path

    return None


def display_video(video_path: str, width: int = 512):
    from base64 import b64encode

    video_data = open(video_path, "rb").read()

    if video_path.lower().endswith(".mp4"):
        mime_type = "video/mp4"
    elif video_path.lower().endswith(".webm"):
        mime_type = "video/webm"
    else:
        mime_type = "video/mp4"

    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()

    display(HTML(f"""
    <video width="{width}" controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))


def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    ensure_dir(output_dir)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path


def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=10, output_dir="/content/ComfyUI/output"):
    ensure_dir(output_dir)
    output_path = f"{output_dir}/{filename_prefix}.webm"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(
        output_path,
        format="FFMPEG",
        mode="I",
        fps=int(fps),
        codec=str(codec),
        quality=int(quality),
        output_params=["-crf", str(int(quality))]
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path


def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    ensure_dir(output_dir)
    output_path = f"{output_dir}/{filename_prefix}.png"
    frame = (image.cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(frame).save(output_path)
    return output_path


# ---------- clip similarity utils ----------
def _ensure_bhwc(img: torch.Tensor):
    return img.unsqueeze(0) if img.dim() == 3 else img


def _extract_vec(cv_out):
    if torch.is_tensor(cv_out):
        v = cv_out
    elif isinstance(cv_out, dict):
        if "image_embeds" in cv_out:
            v = cv_out["image_embeds"]
        elif "pooled_output" in cv_out:
            v = cv_out["pooled_output"]
        elif "last_hidden_state" in cv_out:
            t = cv_out["last_hidden_state"]
            v = t[:, 0] if t.ndim == 3 else t
        else:
            v = next((x for x in cv_out.values() if torch.is_tensor(x)), None)
            if v is None:
                raise ValueError(
                    f"Unknown CLIP vision output keys: {list(cv_out.keys())}")
    else:
        if hasattr(cv_out, "image_embeds"):
            v = getattr(cv_out, "image_embeds")
        elif hasattr(cv_out, "pooled_output"):
            v = getattr(cv_out, "pooled_output")
        elif hasattr(cv_out, "last_hidden_state"):
            t = getattr(cv_out, "last_hidden_state")
            v = t[:, 0] if t.ndim == 3 else t
        else:
            raise ValueError(
                f"Unknown CLIP vision output type: {type(cv_out)}")

    v = v.detach().float()
    if v.ndim > 2:
        v = v.flatten(start_dim=1)
    if v.ndim == 2 and v.shape[0] == 1:
        v = v[0]
    return v


def _cosine(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8):
    a = a.flatten()
    b = b.flatten()
    return float((a @ b) / (a.norm() * b.norm() + eps))


def plot_similarity(times, sims, title="CLIP similarity"):
    plt.figure(figsize=(10, 4))
    plt.plot(times, sims)
    plt.ylim(-0.1, 1.0)
    plt.xlabel("time (s)")
    plt.ylabel("cosine similarity")
    plt.title(title)
    plt.show()


def save_clip_log(clip_log, path="/content/ComfyUI/output/clip_similarity_log.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clip_log, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved clip log: {path}")


def plot_similarity_combined(clip_log, title="All chunks similarity"):
    if not clip_log:
        print("clip_log is empty")
        return

    plt.figure(figsize=(14, 5))
    offset = 0.0
    boundaries = []

    for entry in clip_log:
        prefix = entry["prefix"]
        times = np.array(entry["times"], dtype=float)
        sims = np.array(entry["sims"], dtype=float)

        t_global = times + offset
        plt.plot(t_global, sims)

        end_t = float(t_global[-1]) if len(t_global) else offset
        boundaries.append((end_t, prefix))

        chunk_seconds = entry.get("frames", 0) / max(entry.get("fps", 1), 1)
        offset += chunk_seconds

    for x, prefix in boundaries:
        plt.axvline(x, linestyle="--", linewidth=1, alpha=0.5)
        plt.text(x, 0.02, prefix, rotation=90,
                 va="bottom", ha="right", fontsize=8)

    plt.ylim(-0.1, 1.0)
    plt.xlabel("global time (s)")
    plt.ylabel("cosine similarity")
    plt.title(title)
    plt.show()


def build_anchor_vec_from_image(image_path: str):
    loaded_image = load_image.load_image(image_path)[0]
    clip_vision = clip_vision_loader.load_clip("clip_vision_h.safetensors")[0]
    out = clip_vision_encode.encode(clip_vision, loaded_image, "none")[0]
    anchor_vec = _extract_vec(out).cpu()

    del loaded_image
    del clip_vision
    del out
    clear_memory()
    return anchor_vec


def similarity_for_last_frame(last_frame_path: str, anchor_vec: torch.Tensor):
    loaded_image = load_image.load_image(last_frame_path)[0]
    clip_vision = clip_vision_loader.load_clip("clip_vision_h.safetensors")[0]
    out = clip_vision_encode.encode(clip_vision, loaded_image, "none")[0]
    vec = _extract_vec(out).cpu()
    sim = _cosine(vec, anchor_vec)

    del loaded_image
    del clip_vision
    del out
    clear_memory()
    return sim


# ---------- main generation ----------
def generate_video(
    image_path: str = None,
    positive_prompt: str = "a cute anime girl",
    negative_prompt: str = "",
    width: int = 480,
    height: int = 816,
    seed: int = 0,
    steps: int = 4,
    cfg_scale: float = 1.0,
    sampler_name: str = "euler",
    scheduler: str = "simple",
    frames: int = 61,
    fps: int = 8,
    output_format: str = "mp4",
    filename_prefix: str = "ComfyUI",
    last_frame_dir: str = "/content/ComfyUI/input",
    clip_log=None,
    compute_similarity: bool = False,
):
    output_path = None
    last_frame_path = None

    seed = seed if seed != 0 else random.randint(0, 2**32 - 1)
    print(f"Using seed: {seed}")

    if image_path is None:
        print("Please upload an image file:")
        image_path = upload_image()

    if image_path is None:
        print("No image uploaded!")
        return None, None

    anchor_vec = None
    if compute_similarity:
        print("Preparing similarity anchor...")
        anchor_vec = build_anchor_vec_from_image(image_path)

    with torch.inference_mode():
        print("Loading text encoder...")
        clip = clip_loader.load_clip(
            "umt5_xxl_fp8_e4m3fn_scaled.safetensors",
            "wan",
            "default"
        )[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        clear_memory()

        print("Loading input image...")
        loaded_image = load_image.load_image(image_path)[0]

        print("Loading CLIP Vision for i2v conditioning...")
        clip_vision = clip_vision_loader.load_clip(
            "clip_vision_h.safetensors")[0]
        clip_vision_output = clip_vision_encode.encode(
            clip_vision, loaded_image, "none")[0]

        del clip_vision
        clear_memory()

        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]

        positive_out, negative_out, latent = wan_image_to_video.encode(
            positive,
            negative,
            vae,
            width,
            height,
            frames,
            1,
            loaded_image,
            clip_vision_output
        )

        # самое важное: всё лишнее убираем ДО загрузки UNet
        del loaded_image
        del clip_vision_output
        del positive
        del negative
        clear_memory()

        print("Loading UNet...")
        model = unet_loader.load_unet(unet_name)[0]
        model = model_sampling.patch(model, 5)[0]

        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive_out,
            negative=negative_out,
            latent_image=latent,
        )[0]

        del model
        del positive_out
        del negative_out
        del latent
        clear_memory()

        print("Decoding latents...")
        decoded = vae_decode.decode(vae, sampled)[0]

        del vae
        del sampled
        clear_memory()

        last_frame = decoded[-1] if len(decoded) > 1 else decoded[0]
        last_frame_path = save_as_image(
            last_frame,
            f"{filename_prefix}_lastframe",
            output_dir=last_frame_dir
        )
        print(f"Last frame saved to: {last_frame_path}")

        if len(decoded) == 1:
            output_path = save_as_image(
                decoded[0], filename_prefix, output_dir="/content/ComfyUI/output")
            display(IPImage(filename=output_path))
        else:
            if output_format.lower() == "webm":
                output_path = save_as_webm(decoded, filename_prefix, fps=fps)
            elif output_format.lower() == "mp4":
                output_path = save_as_mp4(decoded, filename_prefix, fps=fps)
            else:
                raise ValueError(f"Unsupported output format: {output_format}")

            display_video(output_path)

        del decoded
        clear_memory()

    # similarity только ПОСЛЕ полной генерации и очистки
    if compute_similarity and anchor_vec is not None and last_frame_path is not None:
        print("Computing similarity on last frame...")
        sim = similarity_for_last_frame(last_frame_path, anchor_vec)
        print(f"Last frame similarity to input image: {sim:.4f}")

        if clip_log is not None:
            clip_log.append({
                "prefix": filename_prefix,
                "times": [0.0],
                "sims": [float(sim)],
                "fps": fps,
                "frames": frames,
                "mode": "last_frame_only"
            })

    clear_memory()
    return output_path, last_frame_path


def run_chunk_loop(
    n_chunks: int = 10,
    start_image_path: str = None,
    same_seed: bool = False,
    positive_prompt: str = "anime girl dancing disco, full body, steady character identity, big red cat dancing next to her, simple clean disco stage, mirror ball, soft neon lighting, gentle camera movement, slow pan left to right, slight zoom in, smooth animation, clear readable motion, consistent outfit, consistent face, no sudden scene changes, no cuts, no flicker, high detail, clean edges, crisp lineart, vibrant but controlled colors",
    negative_prompt: str = "plastic skin, shinny skin, white skin, unnatural skin, skin smoothness, smooth skinwithout testicles, good skin, unnatural skin, shinny skin, skin pores, skin plastic, smooth skins, bad anatomy, morbid anatomy, worst quality, low quality, normal quality, low res, blurry, grainy, pixelated:1.4, deformed, distorted, disfigured, mutation, mutated, bad anatomy, wrong anatomy, inaccurate anatomy, poorly drawn face, poorly drawn hands, missing fingers, extra fingers, fused fingers, extra limbs, missing limbs, cartoon, anime, illustration, 3d, cgi, sketch, painting, drawing, watermark, text, signature, logo, monochrome, b&w",
    width: int = 480,
    height: int = 816,
    seed: int = 0,
    steps: int = 4,
    cfg_scale: float = 1.0,
    sampler_name: str = "euler",
    scheduler: str = "simple",
    frames: int = 61,
    fps: int = 8,
    output_format: str = "mp4",
    compute_similarity: bool = False,
):
    clip_log = []
    videos = []

    if start_image_path is None:
        print("Upload start image:")
        start_image_path = upload_image()
        if start_image_path is None:
            print("No image uploaded!")
            return [], None, []

    current_image = start_image_path

    for i in range(n_chunks):
        print("\n====================")
        print(f"CHUNK {i+1}/{n_chunks}")
        print(f"Input image: {current_image}")
        print("====================\n")

        current_seed = seed if same_seed else random.randint(0, 2**32 - 1)
        print(f"Using seed: {current_seed}")

        prefix = f"video_{i+1:03d}"

        output_path, last_frame_path = generate_video(
            image_path=current_image,
            positive_prompt=positive_prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            seed=current_seed,
            steps=steps,
            cfg_scale=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            frames=frames,
            fps=fps,
            output_format=output_format,
            filename_prefix=prefix,
            clip_log=clip_log,
            compute_similarity=compute_similarity,
        )

        if output_path:
            videos.append(output_path)
            print(f"Saved video: {output_path}")

        if last_frame_path is None:
            print("Failed to get last frame. Stopping loop.")
            return videos, None, clip_log

        current_image = last_frame_path
        clear_memory()

    if compute_similarity:
        save_clip_log(clip_log)
        plot_similarity_combined(clip_log, title="All chunks similarity")

    print(f"\nLoop finished. Last frame for next run: {current_image}")
    return videos, current_image, clip_log

# генерация по сэмплу csv


def generate_from_csv(
    csv_path: str,
    generated_root: str,
    positive_prompt_col: str = "prompt",
    image_col: str = "first_frame_path",
    real_video_col: str = "real_video_path",
    class_col: str = "class",
    video_id_col: str = "video_id",
    width: int = 320,
    height: int = 512,
    steps: int = 4,
    cfg_scale: float = 1.0,
    sampler_name: str = "euler",
    scheduler: str = "simple",
    frames: int = 40,
    fps: int = 8,
    output_format: str = "mp4",
    same_seed: bool = False,
    base_seed: int = 42,
    negative_prompt_value: str = "",
):
    df = pd.read_csv(csv_path)
    generated_root = Path(generated_root)
    generated_root.mkdir(parents=True, exist_ok=True)

    generated_rows = []
    failed_rows = []

    for idx, row in df.iterrows():
        cls = row[class_col]
        video_id = row[video_id_col]
        image_path = row[image_col]
        prompt = row[positive_prompt_col]
        real_video_path = row[real_video_col]

        out_dir = generated_root / cls
        out_dir.mkdir(parents=True, exist_ok=True)

        final_output_path = out_dir / f"{video_id}.mp4"
        prefix = f"tmp_{video_id}"

        try:
            if final_output_path.exists():
                print(f"[skip] exists: {final_output_path}")
            else:
                seed_val = base_seed if same_seed else int(
                    np.random.randint(0, 2**32 - 1))
                print(f"[{idx+1}/{len(df)}] generating {cls} | {video_id}")

                output_path, last_frame_path = generate_video(
                    image_path=image_path,
                    positive_prompt=prompt,
                    negative_prompt=negative_prompt_value,
                    width=width,
                    height=height,
                    seed=seed_val,
                    steps=steps,
                    cfg_scale=cfg_scale,
                    sampler_name=sampler_name,
                    scheduler=scheduler,
                    frames=frames,
                    fps=fps,
                    output_format=output_format,
                    filename_prefix=prefix,
                    clip_log=None,
                    compute_similarity=False,
                )

                if output_path is None:
                    raise RuntimeError("generate_video returned None")

                shutil.copy2(output_path, final_output_path)

            generated_rows.append({
                "video_id": video_id,
                "class": cls,
                "prompt": prompt,
                "first_frame_path": image_path,
                "real_video_path": real_video_path,
                "generated_video_path": str(final_output_path),
            })

        except Exception as e:
            failed_rows.append({
                "video_id": video_id,
                "class": cls,
                "error": str(e),
            })
            print(f"[fail] {video_id}: {e}")

        clear_memory()

    generated_manifest = pd.DataFrame(generated_rows)
    failed_manifest = pd.DataFrame(failed_rows)

    generated_manifest_path = generated_root / "generated_manifest.csv"
    failed_manifest_path = generated_root / "failed_manifest.csv"

    generated_manifest.to_csv(generated_manifest_path, index=False)
    failed_manifest.to_csv(failed_manifest_path, index=False)

    print("\nDone.")
    print("generated:", len(generated_manifest))
    print("failed:", len(failed_manifest))
    print("manifest:", generated_manifest_path)

    return generated_manifest_path, failed_manifest_path

# Новая версия run_chunk_loop с online-метриками
# ===== Updated run_chunk_loop_with_metrics =====


def _check_thresholds(clip_record, thresholds):
    """Returns (passed, failures). spec: number (min) or dict with 'min'/'max' keys."""
    if not thresholds:
        return True, []
    failures = []
    for metric, spec in thresholds.items():
        val = clip_record.get(metric)
        if val is None:
            continue
        try:
            val = float(val)
        except (TypeError, ValueError):
            continue
        if isinstance(spec, dict):
            if "min" in spec and val < spec["min"]:
                failures.append(f"{metric}={val:.4f} < min={spec['min']}")
            if "max" in spec and val > spec["max"]:
                failures.append(f"{metric}={val:.4f} > max={spec['max']}")
        else:
            if val < float(spec):
                failures.append(f"{metric}={val:.4f} < {spec}")
    return len(failures) == 0, failures


def run_chunk_loop_with_metrics(
    image_path,
    positive_prompt,
    negative_prompt,
    num_clips=3,
    width=320,
    height=512,
    steps=4,
    cfg_scale=1.0,
    sampler_name="euler",
    scheduler="simple",
    frames=40,
    fps=8,
    output_format="mp4",
    same_seed=False,
    base_seed=42,

    # старые chunk-consistency метрики
    metrics_out_dir=None,
    metrics_prefix="rollout",
    clips_per_chunk=4,
    frames_per_clip=16,
    metric_resize=(224, 224),
    compute_chunk_metrics=True,

    # Omni-Metric
    world_spec=None,
    omni_overrides=None,
    compute_omni_metrics=True,
    omni_sampling_mode="uniform",
    omni_num_sampled_frames=8,
    omni_custom_frame_indices=None,
    compute_camera_control_flag=True,

    # VBench
    compute_vbench_metrics=True,
    vbench_metric=None,
    vbench_runner=None,
    vbench_dims=None,
    vbench_eval_output_dir="evaluation_results",
    vbench_temp_input_root="/content/vbench_inputs",

    # CSV логирование
    chunk_metrics_csv_path=None,
    rollout_metrics_csv_path=None,

    # full-rollout
    compute_full_rollout_vbench_flag=True,
    full_rollout_output_path=None,

    # full-rollout FVD
    compute_full_rollout_fvd_flag=False,
    # callback(full_rollout_path) -> dict or float
    compute_full_rollout_fvd_fn=None,
    fvd_metric=None,

    # quality gate
    metric_thresholds=None,
    max_retries=3,
    attempt_log_csv_path=None,

    # optional metadata
    sample_id=None,
    sample_class=None,
):
    if vbench_dims is None:
        vbench_dims = VBENCH_CUSTOM_DIMS
    if vbench_metric is None and vbench_runner is not None:
        vbench_metric = vbench_runner

    videos = []
    current_image = image_path
    clip_log = []
    attempts_log = []
    fvd_metric = fvd_metric or FVDMetric(
        device="cuda" if torch.cuda.is_available() else "cpu",
        batch_size=4,
        num_frames=frames_per_clip,
        resize_to=metric_resize,
        sampling_strategy="uniform",
        max_pairs=None,
        cache_features=False,
        cache_dir=None,
        verbose=False,
    )
    metric_history = fvd_metric.init_rollout_metric_history()
    current_world_spec = deepcopy(world_spec) if isinstance(
        world_spec, dict) else None

    clip_idx = 0
    while clip_idx < num_clips:
        output_path = None
        last_frame_path = None
        clip_record = None

        for attempt in range(max_retries + 1):
            _snap_videos = len(videos)
            _snap_dists = list(metric_history["chunk_distributions"])
            _snap_adj = list(metric_history["adjacent_records"])
            _snap_drift = list(metric_history["drift_records"])
            _snap_log = len(clip_log)

            seed_val = base_seed if same_seed else int(
                np.random.randint(0, 2**32 - 1))

            print(f"\n=== Generating clip {clip_idx + 1}/{num_clips} (attempt {attempt + 1}/{max_retries + 1}) ===")

            output_path, last_frame_path = generate_video(
                image_path=current_image,
                positive_prompt=positive_prompt,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                seed=seed_val,
                steps=steps,
                cfg_scale=cfg_scale,
                sampler_name=sampler_name,
                scheduler=scheduler,
                frames=frames,
                fps=fps,
                output_format=output_format,
                filename_prefix=f"clip_{clip_idx:03d}",
                clip_log=None,
                compute_similarity=False,
            )

            if output_path is None:
                print(f"[fail] clip {clip_idx} generation failed")
                break

            videos.append(output_path)

            clip_record = {
                "sample_id": sample_id,
                "class": sample_class,
                "chunk_idx": clip_idx,
                "generated_video_path": output_path,
                "input_image_path": current_image,
                "last_frame_path": last_frame_path,
                "seed": seed_val,
                "attempt": attempt,
            }

            # ---------- existing chunk-consistency metrics ----------
            if compute_chunk_metrics:
                try:
                    chunk_dist = fvd_metric.compute_distribution_for_generated_chunk(
                        chunk_video_path=output_path,
                        clips_per_chunk=clips_per_chunk,
                        frames_per_clip=frames_per_clip,
                        resize=metric_resize,
                    )

                    (
                        metric_history["chunk_distributions"],
                        adjacent_record,
                        drift_record
                    ) = fvd_metric.update_chunk_consistency_history(
                        metric_history["chunk_distributions"],
                        chunk_dist
                    )

                    if adjacent_record is not None:
                        metric_history["adjacent_records"].append(adjacent_record)
                        clip_record["adjacent_cross_chunk_frechet"] = adjacent_record["adjacent_cross_chunk_frechet"]
                    else:
                        clip_record["adjacent_cross_chunk_frechet"] = None

                    metric_history["drift_records"].append(drift_record)
                    clip_record["drift_from_first_chunk"] = drift_record["drift_from_first_chunk"]

                except Exception as e:
                    clip_record["adjacent_cross_chunk_frechet"] = None
                    clip_record["drift_from_first_chunk"] = None
                    clip_record["chunk_metric_error"] = str(e)
                    print(f"[chunk metric fail] chunk {clip_idx}: {e}")

            # ---------- Omni-Metric ----------
            omni_spec = None
            if compute_omni_metrics:
                try:
                    omni_spec = modular_build_omni_spec(
                        world_spec=current_world_spec,
                        chunk_idx=clip_idx,
                        input_image=current_image,
                        generated_video_path=output_path,
                        last_frame_path=last_frame_path,
                        positive_prompt=positive_prompt,
                        fps=fps,
                        num_frames=frames,
                        resolution=(width, height),
                        overrides=omni_overrides,
                    )
                    omni_results = omni_runner.run_on_chunk(
                        omni_spec=omni_spec,
                        generated_video_path=output_path,
                        last_frame_path=last_frame_path,
                        world_spec=current_world_spec,
                        sampling_mode=omni_sampling_mode,
                        max_sampled_frames=omni_num_sampled_frames,
                        custom_indices=omni_custom_frame_indices,
                        compute_camera_control_flag=compute_camera_control_flag,
                    )
                except Exception as e:
                    omni_results = {
                        "interstab_l": None,
                        "interstab_n": None,
                        "intercov": None,
                        "interorder": None,
                        "transitions_detect": None,
                        "object_control": None,
                        "camera_control": None,
                        "omni_status": "failed",
                        "omni_metric_statuses": {},
                        "omni_metric_faithfulness": {},
                        "omni_metric_verification_modes": {},
                        "omni_details": {"status": "failed", "error": str(e)},
                    }
                    print(f"[Omni fail] chunk {clip_idx}: {e}")

                for metric_key, clip_key in OMNI_CLIP_LOG_COLUMNS.items():
                    clip_record[clip_key] = omni_results.get(metric_key)
                clip_record["omni_status"] = omni_results.get(
                    "omni_status", "failed")
                clip_record["omni_details"] = json.dumps(_omni_jsonable(
                    omni_results.get("omni_details", {})), ensure_ascii=False)
                clip_record["omni_metric_statuses"] = json.dumps(_omni_jsonable(
                    omni_results.get("omni_metric_statuses", {})), ensure_ascii=False)
                clip_record["omni_metric_faithfulness"] = json.dumps(_omni_jsonable(
                    omni_results.get("omni_metric_faithfulness", {})), ensure_ascii=False)
                clip_record["omni_metric_verification_modes"] = json.dumps(_omni_jsonable(
                    omni_results.get("omni_metric_verification_modes", {})), ensure_ascii=False)
                if omni_spec is not None:
                    clip_record["omni_target_entities"] = json.dumps(_omni_jsonable(
                        omni_spec.get("target_entities", [])), ensure_ascii=False)
                    clip_record["omni_affected_entities"] = json.dumps(_omni_jsonable(
                        omni_spec.get("affected_entities", [])), ensure_ascii=False)
                    clip_record["omni_unaffected_entities"] = json.dumps(_omni_jsonable(
                        omni_spec.get("unaffected_entities", [])), ensure_ascii=False)
                    clip_record["omni_event_sequence"] = json.dumps(_omni_jsonable(
                        omni_spec.get("event_sequence", [])), ensure_ascii=False)
            else:
                for clip_key in OMNI_CLIP_LOG_COLUMNS.values():
                    clip_record[clip_key] = None
                clip_record["omni_status"] = "skipped"
                skipped_map = {metric_name: "skipped" for metric_name in OMNI_SCORE_KEYS}
                clip_record["omni_details"] = json.dumps(
                    {"status": "skipped", "reason": "disabled_by_flag"}, ensure_ascii=False)
                clip_record["omni_metric_statuses"] = json.dumps(
                    skipped_map, ensure_ascii=False)
                clip_record["omni_metric_faithfulness"] = json.dumps(
                    {metric_name: OMNI_IMPLEMENTATION_STATUS.get(metric_name) for metric_name in OMNI_SCORE_KEYS}, ensure_ascii=False)
                clip_record["omni_metric_verification_modes"] = json.dumps(
                    {metric_name: "skipped" for metric_name in OMNI_SCORE_KEYS}, ensure_ascii=False)

            clip_log.append(clip_record)

            # ---------- save intermediate csv after Omni ----------
            if chunk_metrics_csv_path is not None:
                save_chunk_metrics_csv(clip_log, chunk_metrics_csv_path)

            # ---------- VBench per chunk ----------
            if compute_vbench_metrics and vbench_metric is not None:
                clear_memory()  # free ComfyUI models before loading VBench models
                vbench_result = vbench_metric.run_on_chunk(
                    video_path=output_path,
                    chunk_idx=clip_idx,
                    dimensions=vbench_dims,
                    extra_context={
                        "eval_name_prefix": f"{metrics_prefix}_chunk_{clip_idx:03d}_a{attempt}",
                        "output_dir": vbench_eval_output_dir,
                        "temp_input_root": vbench_temp_input_root,
                        "cleanup_temp_dir": True,
                        "verbose": False,
                    },
                )
                clip_record.update(vbench_result.get("flat_scores", {}))
                if vbench_result.get("status") == "failed":
                    clip_record["vbench_error"] = vbench_result.get("error")
                    print(f"[VBench fail] chunk {clip_idx}: {vbench_result.get('error')}")

            # ---------- save intermediate csv after VBench ----------
            if chunk_metrics_csv_path is not None:
                save_chunk_metrics_csv(clip_log, chunk_metrics_csv_path)


            # ---------- quality gate ----------
            if metric_thresholds:
                _passed, _failures = _check_thresholds(clip_record, metric_thresholds)
                clip_record["threshold_passed"] = _passed
                clip_record["threshold_failures"] = json.dumps(_failures)
                if not _passed and attempt < max_retries:
                    print(f"[retry] chunk {clip_idx} attempt {attempt + 1}/{max_retries + 1}: {_failures}")
                    clip_record["retry_outcome"] = "rejected"
                    attempts_log.append(deepcopy(clip_record))
                    del videos[_snap_videos:]
                    metric_history["chunk_distributions"] = _snap_dists
                    metric_history["adjacent_records"] = _snap_adj
                    metric_history["drift_records"] = _snap_drift
                    del clip_log[_snap_log:]
                    clear_memory()
                    continue
            clip_record["retry_outcome"] = "accepted"
            attempts_log.append(deepcopy(clip_record))
            break  # accepted (or max retries reached)

        if output_path is None:
            break

        previous_image = current_image
        current_image = last_frame_path

        if current_world_spec is not None:
            try:
                current_world_spec = update_world_spec_after_chunk(
                    world_spec=current_world_spec,
                    last_frame_path=last_frame_path,
                    chunk_idx=clip_idx + 1,
                )
            except Exception:
                current_world_spec = deepcopy(current_world_spec)
                current_world_spec["chunk_idx"] = clip_idx + 1
                current_world_spec["previous_chunk_last_frame"] = previous_image
                current_world_spec["current_scene_image"] = str(
                    last_frame_path)

        clear_memory()

        clip_idx += 1
    # ---------- full-rollout summary ----------
    rollout_record = {
        "sample_id": sample_id,
        "class": sample_class,
        "num_chunks": len(videos),
    }

    rollout_record.update(add_optional_full_rollout_omni_summary(clip_log))
    rollout_record.update(summarize_chunk_vbench_metrics(clip_log, dimensions=vbench_dims))

    # ---------- full-rollout VBench ----------
    if compute_full_rollout_vbench_flag and vbench_metric is not None and len(videos) > 0:
        if full_rollout_output_path is None:
            if metrics_out_dir is None:
                metrics_out_dir = "/content/metrics_outputs"
            full_rollout_output_path = str(
                Path(metrics_out_dir) / f"{metrics_prefix}_full_rollout.mp4")

        full_rollout_vbench_result = vbench_metric.run_on_rollout(
            chunk_video_paths=videos,
            full_rollout_output_path=full_rollout_output_path,
            dimensions=vbench_dims,
            extra_context={
                "eval_name_prefix": f"{metrics_prefix}_fullrollout",
                "output_dir": vbench_eval_output_dir,
                "temp_input_root": vbench_temp_input_root,
                "verbose": False,
            },
        )

        if full_rollout_vbench_result.get("full_rollout_path") is not None:
            rollout_record["full_rollout_path"] = full_rollout_vbench_result["full_rollout_path"]
        for k, v in full_rollout_vbench_result.get("flat_scores", {}).items():
            rollout_record[f"fullrollout_{k}"] = v
        if full_rollout_vbench_result.get("status") == "failed":
            rollout_record["full_rollout_vbench_error"] = full_rollout_vbench_result.get("error")
            print(f"[full-rollout VBench fail] {full_rollout_vbench_result.get('error')}")

    # ---------- full-rollout FVD ----------
    if compute_full_rollout_fvd_flag and compute_full_rollout_fvd_fn is not None:
        try:
            fvd_result = compute_full_rollout_fvd_fn(
                rollout_record.get("full_rollout_path", None))
            if isinstance(fvd_result, dict):
                for k, v in fvd_result.items():
                    rollout_record[f"fullrollout_{k}"] = v
            else:
                rollout_record["fullrollout_fvd"] = fvd_result
        except Exception as e:
            rollout_record["full_rollout_fvd_error"] = str(e)
            print(f"[full-rollout FVD fail] {e}")

    # ---------- retry stats ----------
    rejected_attempts = [r for r in attempts_log if r.get("retry_outcome") == "rejected"]
    accepted_attempts = [r for r in attempts_log if r.get("retry_outcome") == "accepted"]
    failure_counts = {}
    for r in rejected_attempts:
        try:
            failures = json.loads(r.get("threshold_failures") or "[]")
        except Exception:
            failures = []
        for f in failures:
            metric = f.split("=")[0].strip()
            failure_counts[metric] = failure_counts.get(metric, 0) + 1
    rollout_record["gate_triggered_count"] = len(rejected_attempts)
    rollout_record["gate_failure_breakdown"] = json.dumps(failure_counts)
    rollout_record["chunks_retried_count"] = len(set(r.get("chunk_idx") for r in rejected_attempts))
    rollout_record["avg_attempts_per_chunk"] = round(
        len(attempts_log) / max(len(accepted_attempts), 1), 2)

    # ---------- save attempt log ----------
    if attempt_log_csv_path is not None:
        save_attempt_log_csv(attempts_log, attempt_log_csv_path)

    # ---------- save rollout csv ----------
    if rollout_metrics_csv_path is not None:
        save_rollout_metrics_csv(rollout_record, rollout_metrics_csv_path)

    return videos, current_image, clip_log, metric_history, rollout_record

## Omni-Metric coverage notes

Current Omni-Metric coverage is aligned to the article definitions as closely as the notebook infrastructure allows.

What is implemented directly in this notebook:
- long-horizon revisit consistency (`InterStab-L`)
- non-target stability via motion energy outside target regions (`InterStab-N`)
- object-level causal faithfulness (`InterCov`)
- event-order verification (`InterOrder`)
- scene transition detection (`Transitions Detect`)
- target-entity preservation (`Object Control`)
- optional camera control when camera expectations are available

What still remains infrastructure-limited:
- detector / segmentation-supervised localization for exact per-entity regions
- VLM / VQA verification for semantic object persistence and event grounding
- trajectory-grounded camera evaluation when no explicit camera trajectory is provided

In those cases the code now returns an explicit `partial` or `skipped` status instead of masking the limitation behind a fake precise score.


In [ ]:
# ===== Example world_spec =====

image_path = "/content/drive/MyDrive/datasets/UCF101_subset/_prepared/first_frames/BenchPress/v_BenchPress_g01_c01_first.png"

world_spec = build_world_spec(
    image_path=image_path,
    next_scene_prompt="a person is doing bench press in a gym",
    camera_prompt="static camera",
    target_objects=["person", "bench", "gym equipment"],
    main_subject="person",
    secondary_subjects=["bench", "gym equipment"],
    expected_action="bench press",
    scene_context="gym interior",
    style_tags=["realistic", "indoor sports scene"],
    consistency_constraints=[
        "keep the same subject identity",
        "keep the same gym environment",
        "keep the bench visible",
        "avoid sudden scene changes",
        "avoid extra people",
    ],
    model_name="wan_i2v_chunked",
    model_type="videogen",
    generate_type="i2v",
    frames_per_chunk=40,
    fps=8,
    resolution=(320, 512),
    num_chunks=3,
    camera_trajectory_gt=None,
    camera_transform_hint=None,
    motion_mask_hint=None,
    expected_scene_transition="continue the same action and preserve the same environment",
    chunk_prompt_schedule={
        0: "a person is doing bench press in a gym",
        1: "a person continues doing bench press in the same gym",
        2: "a person continues the bench press movement in the same gym environment",
    },
)

positive_prompt = build_world_prompt(world_spec)

print("WORLD SPEC:")
print(preview_world_spec(world_spec))
print("\nPOSITIVE PROMPT:")
print(positive_prompt)


In [ ]:
# ===== Extended run_chunk_loop_worldspec =====

def run_chunk_loop_worldspec(
    world_spec: dict,
    negative_prompt: str = "",
    num_clips: int = None,
    width: int = None,
    height: int = None,
    steps: int = 4,
    cfg_scale: float = 1.0,
    sampler_name: str = "euler",
    scheduler: str = "simple",
    frames: int = None,
    fps: int = None,
    output_format: str = "mp4",
    same_seed: bool = False,
    base_seed: int = 42,
):
    videos = []
    chunk_specs = []
    current_spec = deepcopy(world_spec)

    if num_clips is None:
        num_clips = current_spec.get("num_chunks", 3)

    if frames is None:
        frames = current_spec.get("frames_per_chunk", 40)

    if fps is None:
        fps = current_spec.get("fps", 8)

    if width is None or height is None:
        res = current_spec.get("resolution", [320, 512])
        width, height = int(res[0]), int(res[1])

    for clip_idx in range(num_clips):
        current_spec["chunk_idx"] = clip_idx
        current_image = current_spec["current_scene_image"]
        positive_prompt = build_world_prompt(current_spec)

        seed_val = base_seed if same_seed else int(np.random.randint(0, 2**32 - 1))

        print(f"\n=== Generating chunk {clip_idx + 1}/{num_clips} ===")
        print("image:", current_image)
        print("prompt:", positive_prompt)

        output_path, last_frame_path = generate_video(
            image_path=current_image,
            positive_prompt=positive_prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            seed=seed_val,
            steps=steps,
            cfg_scale=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            frames=frames,
            fps=fps,
            output_format=output_format,
            filename_prefix=f"worldspec_clip_{clip_idx:03d}",
            clip_log=None,
            compute_similarity=False,
        )

        if output_path is None:
            print(f"[fail] chunk {clip_idx} generation failed")
            break

        videos.append(output_path)

        chunk_specs.append(
            {
                "chunk_idx": clip_idx,
                "model_name": current_spec.get("model_name"),
                "model_type": current_spec.get("model_type"),
                "generate_type": current_spec.get("generate_type"),
                "input_image": current_image,
                "generated_video_path": output_path,
                "last_frame_path": last_frame_path,
                "positive_prompt": positive_prompt,
                "negative_prompt": negative_prompt,
                "camera_prompt": current_spec.get("camera_prompt"),
                "camera_instruction_text": current_spec.get("camera_instruction_text"),
                "main_subject": current_spec.get("main_subject"),
                "secondary_subjects": current_spec.get("secondary_subjects"),
                "target_objects": current_spec.get("target_objects"),
                "expected_action": current_spec.get("expected_action"),
                "scene_context": current_spec.get("scene_context"),
                "style_tags": current_spec.get("style_tags"),
                "frames_per_chunk": frames,
                "fps": fps,
                "resolution": current_spec.get("resolution"),
                "layout_spec": current_spec.get("layout_spec"),
                "motion_mask_hint": current_spec.get("motion_mask_hint"),
                "camera_trajectory_gt": current_spec.get("camera_trajectory_gt"),
                "camera_transform_hint": current_spec.get("camera_transform_hint"),
                "seed": seed_val,
            }
        )

        current_spec = update_world_spec_after_chunk(
            world_spec=current_spec,
            last_frame_path=last_frame_path,
            chunk_idx=clip_idx + 1,
        )

        clear_memory()

    return videos, chunk_specs, current_spec


In [ ]:
# @title Run Pipeline

import pandas as pd
from pathlib import Path
from src.data.worldscore.adapter import WorldScoreDatasetAdapter

# ---------- Конфиг ----------
SPLIT         = "dynamic"   # "dynamic" | "static"
START_IDX     = 0           # с какого сэмпла начинать
SUBSET_SIZE   = 20          # сколько брать (None = до конца)
NUM_CHUNKS    = 6           # чанков на сэмпл
MAX_RETRIES   = 1
ENABLE_GATE   = True        # False = отключить quality gate (ablation)
NEGATIVE_PROMPT = ""

METRIC_THRESHOLDS = {
    # Omni
    "omni_interstab_l":               {"min": 0.65},
    "omni_interstab_n":               {"min": 0.60},
    "omni_object_control":            {"min": 0.70},
    "omni_camera_control":            {"min": 0.50},
    # VBench
    "vbench_subject_consistency":     {"min": 0.90},
    "vbench_background_consistency":  {"min": 0.88},
    "vbench_motion_smoothness":       {"min": 0.97},
}

# ---------- Пути ----------
DRIVE_ROOT    = Path("/content/drive/MyDrive")
OUTPUTS_ROOT  = DRIVE_ROOT / "worldscore_outputs"
VIDEOS_ROOT   = OUTPUTS_ROOT / "videos"
DATASET_CACHE = OUTPUTS_ROOT / "dataset_cache"
CHUNK_CSV     = OUTPUTS_ROOT / "chunk_metrics.csv"
ROLLOUT_CSV   = OUTPUTS_ROOT / "rollout_metrics.csv"
ATTEMPT_CSV   = OUTPUTS_ROOT / "attempt_log.csv"

for p in [OUTPUTS_ROOT, VIDEOS_ROOT, DATASET_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

# ---------- Resume ----------
# Пропускаем только если num_chunks в CSV >= текущего NUM_CHUNKS
done_ids = set()
if ROLLOUT_CSV.exists():
    df_done = pd.read_csv(ROLLOUT_CSV)
    if not df_done.empty:
        if "num_chunks" in df_done.columns:
            done_ids = set(df_done[df_done["num_chunks"] >= NUM_CHUNKS]["sample_id"].tolist())
            rerun_count = df_done["sample_id"].nunique() - len(done_ids)
        else:
            done_ids = set(df_done["sample_id"].tolist())
            rerun_count = 0
        print(f"Resume: {len(done_ids)} готовых (≥{NUM_CHUNKS} чанков) пропускаем, {rerun_count} перезапустим")

# ---------- VBench (один раз на весь цикл) ----------
vbench_metric = VBenchMetric(
    dimensions=VBENCH_CUSTOM_DIMS,
    device="cuda" if torch.cuda.is_available() else "cpu",
    full_info_json="/content/VBench/vbench/VBench_full_info.json",
    output_dir="evaluation_results",
    repo_root="/content/VBench",
    temp_input_root="/content/vbench_inputs",
    verbose=False,
)

# ---------- Датасет ----------
adapter = WorldScoreDatasetAdapter(
    split_name=SPLIT,
    cache_dir=str(DATASET_CACHE),
    frames_per_chunk=25,
    fps=8,
    resolution=(320, 512),
    num_chunks=NUM_CHUNKS,
)

indices = list(range(len(adapter)))
if SUBSET_SIZE is not None:
    indices = indices[START_IDX : START_IDX + SUBSET_SIZE]
else:
    indices = indices[START_IDX:]

print(f"Датасет: {SPLIT} | всего {len(adapter)} | запускаем {len(indices)}")

# ---------- Основной цикл ----------
errors = []
for i, idx in enumerate(indices):
    sample    = adapter.get_sample(idx)
    sample_id = sample.sample_id

    if sample_id in done_ids:
        print(f"[{i+1}/{len(indices)}] {sample_id} — пропуск (уже {NUM_CHUNKS}+ чанков)")
        continue

    target_objs = sample.world_spec.get("target_objects", [])
    motion_type = sample.metadata.get("motion_type") or SPLIT
    print(f"\n[{i+1}/{len(indices)}] {sample_id} | {motion_type} | {target_objs}")

    # Дополняем world_spec полями для Omni-метрик
    ws = sample.world_spec
    ws["affected_entities"]   = target_objs
    ws["unaffected_entities"] = []
    ws["event_sequence"]      = []

    sample_dir  = VIDEOS_ROOT / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    rollout_mp4 = sample_dir / "full_rollout.mp4"

    try:
        videos, _, clip_log, _, _ = run_chunk_loop_with_metrics(
            image_path=sample.image_path,
            positive_prompt=sample.positive_prompt,
            negative_prompt=sample.negative_prompt or NEGATIVE_PROMPT,
            world_spec=ws,
            num_clips=NUM_CHUNKS,
            width=320,
            height=512,
            steps=4,
            cfg_scale=1.0,
            sampler_name="euler",
            scheduler="simple",
            frames=25,
            fps=8,
            output_format="mp4",

            metrics_out_dir=str(OUTPUTS_ROOT),
            metrics_prefix="ws",
            clips_per_chunk=4,
            frames_per_clip=16,
            metric_resize=(224, 224),
            compute_chunk_metrics=True,

            compute_omni_metrics=True,
            omni_sampling_mode="uniform",
            omni_num_sampled_frames=8,

            compute_vbench_metrics=True,
            vbench_metric=vbench_metric,
            vbench_dims=VBENCH_CUSTOM_DIMS,
            vbench_eval_output_dir="evaluation_results",
            vbench_temp_input_root="/content/vbench_inputs",

            chunk_metrics_csv_path=str(CHUNK_CSV),
            rollout_metrics_csv_path=str(ROLLOUT_CSV),

            compute_full_rollout_vbench_flag=True,
            full_rollout_output_path=str(rollout_mp4),

            compute_full_rollout_fvd_flag=False,
            fvd_metric=fvd_metric,

            metric_thresholds=METRIC_THRESHOLDS if ENABLE_GATE else None,
            max_retries=MAX_RETRIES,
            attempt_log_csv_path=str(ATTEMPT_CSV),

            sample_id=sample_id,
            sample_class=motion_type,
        )
        done_ids.add(sample_id)
        print(f"  ✓ {len(videos)} чанков")

        # Сохраняем eval JSONs на Drive и чистим локальную папку
        local_eval_dir = Path("evaluation_results")
        if local_eval_dir.exists():
            drive_eval_dir = OUTPUTS_ROOT / "evaluation_results" / sample_id
            drive_eval_dir.mkdir(parents=True, exist_ok=True)
            saved = 0
            for f in local_eval_dir.glob("*.json"):
                shutil.copy2(f, drive_eval_dir / f.name)
                saved += 1
            shutil.rmtree(local_eval_dir)
            print(f"  → {saved} eval JSON(s) → {drive_eval_dir}")

    except Exception as e:
        print(f"  ✗ ОШИБКА [{sample_id}]: {e}")
        errors.append({"sample_id": sample_id, "error": str(e)})
        continue

# ---------- Итог ----------
print(f"\nГотово: {len(done_ids)} сэмплов")
if errors:
    print(f"Ошибок: {len(errors)}")
    for err in errors:
        print(f"  {err['sample_id']}: {err['error']}")
print(f"Chunk metrics:   {CHUNK_CSV}")
print(f"Rollout summary: {ROLLOUT_CSV}")
if ROLLOUT_CSV.exists():
    display(pd.read_csv(ROLLOUT_CSV).tail(10))
